In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [33]:
file_path=r"..\data\raw\healthcare-dataset-stroke-data.csv"
data=pd.read_csv(file_path)

data.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


2️⃣ Handle missing values

In [34]:
# Check missing values
print("Missing Value")
print(data.isnull().sum())

Missing Value
id                     0
gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_type              0
Residence_type         0
avg_glucose_level      0
bmi                  201
smoking_status         0
stroke                 0
dtype: int64


In [35]:
# Replace missing 'bmi' with median
data['bmi'].fillna(data['bmi'].median(), inplace=True)
data['smoking_status'].fillna('Unknown')

# print("Filled missing 'bmi' with median and 'smoking_status' with 'Unknown'.")
print("----------------------------------")
print("\nMissing values after processing:")
print(data.isnull().sum())

----------------------------------

Missing values after processing:
id                   0
gender               0
age                  0
hypertension         0
heart_disease        0
ever_married         0
work_type            0
Residence_type       0
avg_glucose_level    0
bmi                  0
smoking_status       0
stroke               0
dtype: int64


C:\Users\hp\AppData\Local\Temp\ipykernel_15540\1352365336.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['bmi'].fillna(data['bmi'].median(), inplace=True)


3️⃣ Categorical Encoding (temporary numeric encoding)



In [ ]:
# Gender: Male=0, Female=1, Other=2
data['gender'] = data['gender'].map({'Male':0, 'Female':1, 'Other':2})

# Ever_married: No=0, Yes=1
data['ever_married'] = data['ever_married'].map({'No':0, 'Yes':1})

# Residence_type: Rural=0, Urban=1
data['Residence_type'] = data['Residence_type'].map({'Rural':0, 'Urban':1})

# Work_type: one-hot encoding
if 'work_type' in data.columns:
    data = pd.get_dummies(data, columns=['work_type'], prefix='work')
else:
    print("Column 'work_type' not found! Check spelling or previous steps.")

# Smoking_status: one-hot encoding
if 'smoking_status' in data.columns:
    data = pd.get_dummies(data, columns=['smoking_status'], prefix='work')
else:
    print("Column 'work_type' not found! Check spelling or previous steps.")


In [37]:
data.head()

,id,gender,age,hypertension,heart_disease,ever_married,Residence_type,avg_glucose_level,bmi,stroke,work_Govt_job,work_Never_worked,work_Private,work_Self-employed,work_children,work_Unknown,work_formerly smoked,work_never smoked,work_smokes
0,9046,0,67.0,0,1,1,1,228.69,36.6,1,False,False,True,False,False,False,True,False,False
1,51676,1,61.0,0,0,1,0,202.21,28.1,1,False,False,False,True,False,False,False,True,False
2,31112,0,80.0,0,1,1,0,105.92,32.5,1,False,False,True,False,False,False,False,True,False
3,60182,1,49.0,0,0,1,1,171.23,34.4,1,False,False,True,False,False,False,False,False,True
4,1665,1,79.0,1,0,1,0,174.12,24.0,1,False,False,False,True,False,False,False,True,False


4️⃣ Scaling / Normalization

In [40]:
# Normalize numeric features for Fuzzy Logic input
numeric_features = ['age', 'avg_glucose_level', 'bmi']
scaler = MinMaxScaler()
data[numeric_features] = scaler.fit_transform(data[numeric_features])

# Display first 5 rows after scaling
data.head()


,id,gender,age,hypertension,heart_disease,ever_married,Residence_type,avg_glucose_level,bmi,stroke,work_Govt_job,work_Never_worked,work_Private,work_Self-employed,work_children,work_Unknown,work_formerly smoked,work_never smoked,work_smokes
0,9046,0,0.816895,0,1,1,1,0.801265,0.301260,1,False,False,True,False,False,False,True,False,False
1,51676,1,0.743652,0,0,1,0,0.679023,0.203895,1,False,False,False,True,False,False,False,True,False
2,31112,0,0.975586,0,1,1,0,0.234512,0.254296,1,False,False,True,False,False,False,False,True,False
3,60182,1,0.597168,0,0,1,1,0.536008,0.276060,1,False,False,True,False,False,False,False,False,True
4,1665,1,0.963379,1,0,1,0,0.549349,0.156930,1,False,False,False,True,False,False,False,True,False


5️⃣ Target Transformation (optional for Fuzzy)



In [41]:
# For Fuzzy Logic, we can create a stroke risk category
# Example: Low Risk = 0, High Risk = 1 (binary), or Low/Medium/High based on thresholds

# For now, keep binary stroke column for training
data['stroke'].value_counts()


stroke
0    4861
1     249
Name: count, dtype: int64

Fuzzy Risk

In [48]:
data.columns

Index(['id', 'gender', 'age', 'hypertension', 'heart_disease', 'ever_married',
       'Residence_type', 'avg_glucose_level', 'bmi', 'stroke', 'work_Govt_job',
       'work_Never_worked', 'work_Private', 'work_Self-employed',
       'work_children', 'work_Unknown', 'work_formerly smoked',
       'work_never smoked', 'work_smokes', 'stroke_risk'],
      dtype='object')

In [50]:
# Initialize a new column
data['stroke_risk'] = 0  # default = Low risk   1 = M

# Rule-based assignment
# High risk if stroke=1 OR age>70 OR glucose>140
data.loc[(data['stroke']==1) | (data['age']>0.85) | (data['avg_glucose_level']>0.8), 'stroke_risk'] = 2  # High

# Medium risk if age between 50-70 or glucose moderate (0.6-0.8)
data.loc[((data['age']>0.6) & (data['age']<=0.85)) | ((data['avg_glucose_level']>0.6) & (data['avg_glucose_level']<=0.8)), 'stroke_risk'] = 1  # Medium

# Low risk = everything else (already 0)


In [51]:
print(data['stroke_risk'].value_counts())


stroke_risk
0    2775
1    1683
2     652
Name: count, dtype: int64


In [52]:
# Save processed dataset for modeling
data.to_csv('../data/processed/healthcare_processed.csv', index=False)

# save as pickle (faster to load)
data.to_pickle('../data/processed/healthcare_processed.pkl')